In [0]:
%pip install databricks-cli

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:

# In a notebook cell - create scope via API
import requests
import json

workspace_url = "https://dbc-00781aa6-52b5.cloud.databricks.com/" #f"https://{w.config.host}"
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Create secret scope
create_scope_url = f"{workspace_url}/api/2.0/secrets/scopes/create"
response = requests.post(
    create_scope_url,
    headers={"Authorization": f"Bearer {token}"},
    json={
        "scope": "agentic",
        "initial_manage_principal": "users"  # Who can manage this scope
    }
)
print(f"Scope creation: {response.status_code}")

# Store secret (requires CLI - API doesn't support direct secret creation for security)
print("\n⚠️ To store the actual secret value, use CLI:")
print("databricks secrets put --scope rag_model_serving --key databricks_token")

Scope creation: 200

⚠️ To store the actual secret value, use CLI:
databricks secrets put --scope rag_model_serving --key databricks_token


In [0]:
# Configure databricks CLI using environment variables
import os

# Set Databricks configuration
workspace_url = "https://dbc-00781aa6-52b5.cloud.databricks.com"
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Configure databricks CLI
os.environ['DATABRICKS_HOST'] = workspace_url
os.environ['DATABRICKS_TOKEN'] = token

print("✓ Databricks CLI configured successfully")
print(f"Workspace: {workspace_url}")

✓ Databricks CLI configured successfully
Workspace: https://dbc-00781aa6-52b5.cloud.databricks.com


In [0]:
# Store service principal secrets in the scope
import subprocess
import getpass

scope_name = "agentic"

# Prompt for service principal credentials
print("Enter Service Principal credentials:")
print("\n⚠️  These values will be stored securely in Databricks secrets")
print("="*60)

# Get client ID
client_id = input("Enter Service Principal Client ID (Application ID): ").strip()

# Get client secret (hidden input)
client_secret = getpass.getpass("Enter Service Principal Client Secret: ").strip()

# Get tenant ID (if using Azure)
tenant_id = input("Enter Tenant ID (press Enter to skip if not Azure): ").strip()

print("\n" + "="*60)
print("Storing secrets...\n")

# Store client_id using --string-value to avoid vim editor
result = subprocess.run(
    ["databricks", "secrets", "put", "--scope", scope_name, "--key", "sp_client_id", "--string-value", client_id],
    capture_output=True,
    text=True
)
if result.returncode == 0:
    print("✓ Stored sp_client_id")
else:
    print(f"✗ Error storing sp_client_id: {result.stderr}")

# Store client_secret using --string-value to avoid vim editor
result = subprocess.run(
    ["databricks", "secrets", "put", "--scope", scope_name, "--key", "sp_client_secret", "--string-value", client_secret],
    capture_output=True,
    text=True
)
if result.returncode == 0:
    print("✓ Stored sp_client_secret")
else:
    print(f"✗ Error storing sp_client_secret: {result.stderr}")

# Store tenant_id if provided
if tenant_id:
    result = subprocess.run(
        ["databricks", "secrets", "put", "--scope", scope_name, "--key", "sp_tenant_id", "--string-value", tenant_id],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✓ Stored sp_tenant_id")
    else:
        print(f"✗ Error storing sp_tenant_id: {result.stderr}")

print("\n" + "="*60)
print("✓ Service Principal secrets stored successfully!")
print(f"\nScope: {scope_name}")
print("Keys stored:")
print("  - sp_client_id")
print("  - sp_client_secret")
if tenant_id:
    print("  - sp_tenant_id")

Enter Service Principal credentials:

⚠️  These values will be stored securely in Databricks secrets


Enter Service Principal Client ID (Application ID):  [REDACTED]

Enter Service Principal Client Secret:  [REDACTED]

Enter Tenant ID (press Enter to skip if not Azure):  


Storing secrets...

✓ Stored sp_client_id
✓ Stored sp_client_secret

✓ Service Principal secrets stored successfully!

Scope: agentic
Keys stored:
  - sp_client_id
  - sp_client_secret


In [0]:
# How to retrieve and use the service principal secrets
import subprocess

scope_name = "agentic"

# First, list secrets in the scope to verify they exist
print("Checking secrets in scope...\n")
result = subprocess.run(
    ["databricks", "secrets", "list", "--scope", scope_name],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("Available secrets:")
    print(result.stdout)
    print()
else:
    print(f"⚠️  Could not list secrets: {result.stderr}")
    print("\n⚠️  Note: Cell 5 failed to store secrets due to CLI command error.")
    print("Please run Cell 5 again after it's been fixed to use the correct command.")
    print("\nOr store secrets manually using:")
    print(f"  databricks secrets put --scope {scope_name} --key sp_client_id")
    print(f"  databricks secrets put --scope {scope_name} --key sp_client_secret")
    raise Exception("Secrets not found. Please store them first using Cell 5.")

# Try to retrieve secrets
try:
    client_id = dbutils.secrets.get(scope=scope_name, key="sp_client_id")
    client_secret = dbutils.secrets.get(scope=scope_name, key="sp_client_secret")
    print("✓ Retrieved sp_client_id")
    print("✓ Retrieved sp_client_secret")
    
    # If you stored tenant_id
    try:
        tenant_id = dbutils.secrets.get(scope=scope_name, key="sp_tenant_id")
        print("✓ Retrieved sp_tenant_id")
    except:
        tenant_id = None
        print("ℹ️ sp_tenant_id not found (optional)")
    
    print("\n⚠️  Note: Secret values are redacted and won't be displayed")
    print("\n" + "="*60)
    print("Example: Use secrets for authentication")
    print("="*60)
    print("""
# Azure Service Principal Authentication
from azure.identity import ClientSecretCredential

credential = ClientSecretCredential(
    tenant_id=dbutils.secrets.get(scope="agentic", key="sp_tenant_id"),
    client_id=dbutils.secrets.get(scope="agentic", key="sp_client_id"),
    client_secret=dbutils.secrets.get(scope="agentic", key="sp_client_secret")
)

# Or for Databricks API calls
import requests

headers = {
    "Authorization": f"Bearer {dbutils.secrets.get(scope='agentic', key='sp_client_secret')}"
}
""")
except Exception as e:
    print(f"\n❌ Error retrieving secrets: {e}")
    print("\n⚠️  The secrets don't exist yet. Please run Cell 5 to store them first.")

Checking secrets in scope...

Available secrets:
Key name            Last updated
----------------  --------------
sp_client_id       1788657586578
sp_client_secret   1788657587318


✓ Retrieved sp_client_id
✓ Retrieved sp_client_secret
ℹ️ sp_tenant_id not found (optional)

⚠️  Note: Secret values are redacted and won't be displayed

Example: Use secrets for authentication

# Azure Service Principal Authentication
from azure.identity import ClientSecretCredential

credential = ClientSecretCredential(
    tenant_id=dbutils.secrets.get(scope="agentic", key="sp_tenant_id"),
    client_id=dbutils.secrets.get(scope="agentic", key="sp_client_id"),
    client_secret=dbutils.secrets.get(scope="agentic", key="sp_client_secret")
)

# Or for Databricks API calls
import requests

headers = {
    "Authorization": f"Bearer {dbutils.secrets.get(scope='agentic', key='sp_client_secret')}"
}



In [0]:
# Example: Authenticate using Databricks Service Principal
from databricks.sdk import WorkspaceClient
import requests
import os

scope_name = "agentic"
workspace_host = "https://dbc-00781aa6-52b5.cloud.databricks.com"

# Method 1: Using Databricks SDK with OAuth M2M (Machine-to-Machine)
print("Method 1: Databricks SDK with Service Principal OAuth")
print("="*60)

# Temporarily clear environment variables to avoid auth conflict
old_token = os.environ.pop('DATABRICKS_TOKEN', None)

try:
    # Initialize WorkspaceClient with service principal credentials
    w = WorkspaceClient(
        host=workspace_host,
        client_id=dbutils.secrets.get(scope=scope_name, key="sp_client_id"),
        client_secret=dbutils.secrets.get(scope=scope_name, key="sp_client_secret")
    )
    
    print("✓ Authenticated with Databricks service principal")
    try:
        current_user = w.current_user.me()
        print(f"Service Principal: {current_user.user_name}")
    except Exception as e:
        print(f"Note: {e}")
finally:
    # Restore the environment variable
    if old_token:
        os.environ['DATABRICKS_TOKEN'] = old_token

print("\n" + "="*60)
print("Method 2: Direct API calls with OAuth token")
print("="*60)

import requests

# Get OAuth token
client_id = dbutils.secrets.get(scope=scope_name, key="sp_client_id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp_client_secret")

# Exchange credentials for access token
token_url = "https://dbc-00781aa6-52b5.cloud.databricks.com/oidc/v1/token"
response = requests.post(
    token_url,
    data={
        "grant_type": "client_credentials",
        "scope": "all-apis"
    },
    auth=(client_id, client_secret)
)

if response.status_code == 200:
    access_token = response.json()["access_token"]
    print("✓ Successfully obtained OAuth access token")
    
    # Use the token for API calls
    headers = {"Authorization": f"Bearer {access_token}"}
    
    # Example: List clusters
    clusters_response = requests.get(
        "https://dbc-00781aa6-52b5.cloud.databricks.com/api/2.0/clusters/list",
        headers=headers
    )
    print(f"✓ API call successful: {clusters_response.status_code}")
else:
    print(f"✗ Failed to get token: {response.status_code}")
    print(response.text)

print("\n" + "="*60)
print("Common Use Cases:")
print("="*60)
print("""
1. Automated Jobs & Workflows:
   - Run notebooks/jobs as a service principal
   - Programmatic workspace management

2. CI/CD Pipelines:
   - Deploy notebooks, jobs, and clusters
   - Manage Unity Catalog objects

3. External Applications:
   - Query data from external apps
   - Trigger job runs programmatically

4. Cross-workspace operations:
   - Service principals can access multiple workspaces
   - Centralized automation and monitoring
""")

Method 1: Databricks SDK with Service Principal OAuth
✓ Authenticated with Databricks service principal
Service Principal: [REDACTED]

Method 2: Direct API calls with OAuth token
✓ Successfully obtained OAuth access token
✓ API call successful: 200

Common Use Cases:

1. Automated Jobs & Workflows:
   - Run notebooks/jobs as a service principal
   - Programmatic workspace management

2. CI/CD Pipelines:
   - Deploy notebooks, jobs, and clusters
   - Manage Unity Catalog objects

3. External Applications:
   - Query data from external apps
   - Trigger job runs programmatically

4. Cross-workspace operations:
   - Service principals can access multiple workspaces
   - Centralized automation and monitoring



In [0]:
# Troubleshoot Service Principal Authentication
import requests

scope_name = "agentic"
workspace_host = "https://dbc-00781aa6-52b5.cloud.databricks.com"

print("Service Principal Authentication Diagnostics")
print("="*60)

# Step 1: Verify secrets exist and are retrievable
print("\n1. Checking if secrets are stored correctly...")
try:
    client_id = dbutils.secrets.get(scope=scope_name, key="sp_client_id")
    client_secret = dbutils.secrets.get(scope=scope_name, key="sp_client_secret")
    print("   ✓ Secrets retrieved successfully")
    print(f"   ✓ Client ID length: {len(client_id)} characters")
    print(f"   ✓ Client Secret length: {len(client_secret)} characters")
    
    # Check if values look valid (not empty, reasonable length)
    if len(client_id) < 10:
        print("   ⚠️  Client ID seems too short - verify it's correct")
    if len(client_secret) < 10:
        print("   ⚠️  Client Secret seems too short - verify it's correct")
except Exception as e:
    print(f"   ✗ Error retrieving secrets: {e}")
    print("   → Run Cell 5 to store credentials")

# Step 2: Test OAuth endpoint
print("\n2. Testing OAuth token endpoint...")
token_url = f"{workspace_host}/oidc/v1/token"
try:
    # Try with credentials
    response = requests.post(
        token_url,
        data={
            "grant_type": "client_credentials",
            "scope": "all-apis"
        },
        auth=(client_id, client_secret),
        timeout=10
    )
    
    print(f"   Response status: {response.status_code}")
    
    if response.status_code == 200:
        print("   ✓ Authentication successful!")
        token_data = response.json()
        print(f"   ✓ Token type: {token_data.get('token_type')}")
        print(f"   ✓ Expires in: {token_data.get('expires_in')} seconds")
    elif response.status_code == 401:
        print("   ✗ Authentication failed (401 Unauthorized)")
        error_data = response.json()
        print(f"   Error: {error_data.get('error')}")
        print(f"   Description: {error_data.get('error_description')}")
        print("\n   Common causes:")
        print("   • Incorrect Client ID or Client Secret")
        print("   • Service Principal not enabled for OAuth")
        print("   • Service Principal has been deleted or disabled")
    else:
        print(f"   ✗ Unexpected status code: {response.status_code}")
        print(f"   Response: {response.text}")
except Exception as e:
    print(f"   ✗ Error during authentication test: {e}")

print("\n" + "="*60)
print("Next Steps:")
print("="*60)
print("""
If authentication failed, verify in Databricks UI:

1. Go to Settings → Identity and access → Service principals
2. Find your service principal
3. Verify:
   ✓ Service principal is active (not disabled)
   ✓ OAuth is enabled (check 'Enable OAuth' setting)
   ✓ Client ID matches what you stored in secrets
   ✓ Client secret is valid (if unsure, generate a new one)

4. If you need to update credentials:
   • Generate a new client secret in the UI
   • Re-run Cell 5 with the new credentials
   • Run this diagnostic cell again to verify

5. Permissions needed:
   • The service principal needs appropriate workspace permissions
   • For API access, grant 'workspace access' permission
""")

Service Principal Authentication Diagnostics

1. Checking if secrets are stored correctly...
   ✓ Secrets retrieved successfully
   ✓ Client ID length: 36 characters
   ✓ Client Secret length: 36 characters

2. Testing OAuth token endpoint...
   Response status: 200
   ✓ Authentication successful!
   ✓ Token type: Bearer
   ✓ Expires in: 3600 seconds

Next Steps:

If authentication failed, verify in Databricks UI:

1. Go to Settings → Identity and access → Service principals
2. Find your service principal
3. Verify:
   ✓ Service principal is active (not disabled)
   ✓ OAuth is enabled (check 'Enable OAuth' setting)
   ✓ Client ID matches what you stored in secrets
   ✓ Client secret is valid (if unsure, generate a new one)

4. If you need to update credentials:
   • Generate a new client secret in the UI
   • Re-run Cell 5 with the new credentials
   • Run this diagnostic cell again to verify

5. Permissions needed:
   • The service principal needs appropriate workspace permissions
   